# Classificatore basato su SVM lineari e n-grammi.
Questo notebook implementa e valuta un sistema di classificazione testuale basato su **Linear Support Vector Machines (LinearSVC)**. Vengono testate diverse configurazioni di estrazione delle feature (TF-IDF) variando il livello di granularità (forme superficiali, lemmi, Part-of-Speech e caratteri) e l'ampiezza degli n-grammi.

In [14]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

## 1. Caricamento e Ispezione dei Dati

In questa sezione carichiamo i dataset pre-elaborati (Training, Validation e Test set) e ne verifichiamo la struttura interna.

In [15]:
# Caricamento dei dataframe processati
df_train = pd.read_pickle("../data/processed/df_train_processed.pkl")
df_val = pd.read_pickle("../data/processed/df_val_processed.pkl")  
df_test = pd.read_pickle("../data/processed/df_test_processed.pkl")

# Estrazione dei target vector
y_train = df_train['label'].copy()
y_val = df_val['label'].copy()
y_test = df_test['label'].copy()

print(f"Dimensioni Train: {df_train.shape}")
print(f"Dimensioni Validation: {df_val.shape}")
print(f"Dimensioni Test: {df_test.shape}")

Dimensioni Train: (2000, 8)
Dimensioni Validation: (1000, 8)
Dimensioni Test: (1000, 8)


In [16]:
df_train.head()

,text,label,tokens,pos,lemmas,tokens_processed,pos_processed,lemmas_processed
0,"Eppure, nel mezzo di queste scorse commerciali...",1,"[eppure, nel, mezzo, di, queste, scorse, comme...","[CCONJ, ADP, NOUN, ADP, DET, NOUN, ADJ, DET, A...","[eppure, in il, mezzo, di, questo, scorsa, com...",eppure nel mezzo di queste scorse commerciali ...,CCONJ ADP NOUN ADP DET NOUN ADJ DET ADJ NOUN A...,eppure in il mezzo di questo scorsa commercial...
1,"""È una cosa che sto valutando di fare da fine ...",0,"[è, una, cosa, che, sto, valutando, di, fare, ...","[AUX, DET, NOUN, PRON, AUX, VERB, ADP, VERB, A...","[essere, uno, cosa, che, stare, valutare, di, ...",è una cosa che sto valutando di fare da fine a...,AUX DET NOUN PRON AUX VERB ADP VERB ADP NOUN N...,essere uno cosa che stare valutare di fare da ...
2,Il caso Pirelli si presenta non come un sempli...,1,"[il, caso, pirelli, si, presenta, non, come, u...","[DET, NOUN, PROPN, PRON, VERB, ADV, ADP, DET, ...","[il, caso, pirelli, si, presentare, non, come,...",il caso pirelli si presenta non come un sempli...,DET NOUN PROPN PRON VERB ADV ADP DET ADJ NOUN ...,il caso pirelli si presentare non come uno sem...
3,Poi la direzione distrettuale antimafia di Lec...,0,"[poi, la, direzione, distrettuale, antimafia, ...","[ADV, DET, NOUN, ADJ, ADJ, ADP, PROPN, ADP, DE...","[poi, il, direzione, distrettuale, antimafia, ...",poi la direzione distrettuale antimafia di lec...,ADV DET NOUN ADJ ADJ ADP PROPN ADP DET NOUN PR...,poi il direzione distrettuale antimafia di lec...
4,Il cambio di programma si è configurato in una...,1,"[il, cambio, di, programma, si, è, configurato...","[DET, NOUN, ADP, NOUN, PRON, AUX, VERB, ADP, D...","[il, cambio, di, programma, si, essere, config...",il cambio di programma si è configurato in una...,DET NOUN ADP NOUN PRON AUX VERB ADP DET NOUN V...,il cambio di programma si essere configurare i...


## 2. Definizione delle Configurazioni (Feature Engineering)

Definiamo un dizionario di configurazioni per esplorare l'impatto di diversi tipi di tokenizzazione e configurazioni di n-grammi sul calcolo dei punteggi TF-IDF. Sperimenteremo su:
1. **Forme (Tokens)**: Unigrammi e Bigrammi.
2. **Lemmi**: Unigrammi e Bigrammi (per astrarre dalla morfologia).
3. **Part-of-Speech (POS)**: Bigrammi e Trigrammi (per catturare pattern sintattici).
4. **Caratteri**: N-grammi da 3 a 5 caratteri (per catturare pattern sub-lessicali).

In [17]:
configs = {
    # 1. ESPERIMENTI SULLE FORME (TOKENS)
    "forme_unigrammi": {
        "colonna": "tokens_processed",
        "analyzer": "word",
        "ngram_range": (1, 1)
    },
    "forme_unigrammi_bigrammi": {
        "colonna": "tokens_processed",
        "analyzer": "word",
        "ngram_range": (1, 2)
    },
    
    # 2. ESPERIMENTI SUI LEMMI
    "lemmi_unigrammi": {
        "colonna": "lemmas_processed",
        "analyzer": "word",
        "ngram_range": (1, 1)
    },
    "lemmi_bigrammi": {
        "colonna": "lemmas_processed",
        "analyzer": "word",
        "ngram_range": (2, 2)
    },
    
    # 3. ESPERIMENTI SULLE PART-OF-SPEECH (SINTASSI)
    "pos_bigrammi": {
        "colonna": "pos_processed",
        "analyzer": "word",
        "ngram_range": (2, 2)
    },
    "pos_trigrammi": {
        "colonna": "pos_processed",
        "analyzer": "word",
        "ngram_range": (3, 3)
    },
    
    # 4. ESPERIMENTI SUI CARATTERI
    # Nota: per i caratteri usiamo la colonna del testo originale 'text'
    "caratteri_3_5_grammi": {
        "colonna": "text",
        "analyzer": "char",
        "ngram_range": (3, 5)
    }
}

## 3. Model Selection tramite 5-Fold Cross-Validation

Utilizziamo una procedura di **Stratified 5-Fold Cross-Validation** sul solo Training Set per confrontare le performance macro-F1 di ciascuna configurazione ed evitare problemi di data leakage.

In [18]:
for nome_test, params in configs.items():

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer=params["analyzer"],
            ngram_range=params["ngram_range"]
        )),
        ("clf", LinearSVC())
    ])

    scores = cross_val_score(
        pipeline,
        df_train[params["colonna"]],
        y_train,
        cv=5,
        scoring="f1_macro"
    )

    print("\nEsecuzione:", nome_test)
    print("Scores fold:", scores)
    print("Media Macro-F1:", np.mean(scores))


Esecuzione: forme_unigrammi
Scores fold: [0.969988   0.97248607 0.97249158 0.98749805 0.97249983]
Media Macro-F1: 0.9749927033020546

Esecuzione: forme_unigrammi_bigrammi
Scores fold: [0.95996397 0.9774886  0.96998124 0.98499662 0.97249158]
Media Macro-F1: 0.9729844018471552

Esecuzione: lemmi_unigrammi
Scores fold: [0.96246035 0.96748354 0.96498599 0.98249464 0.95498199]
Media Macro-F1: 0.9664813026925627

Esecuzione: lemmi_bigrammi
Scores fold: [0.95492788 0.96246035 0.96247162 0.96999325 0.95997498]
Media Macro-F1: 0.9619656170733745

Esecuzione: pos_bigrammi
Scores fold: [0.93247932 0.919998   0.92249952 0.92249952 0.91749948]
Media Macro-F1: 0.9229951674716041

Esecuzione: pos_trigrammi
Scores fold: [0.94249101 0.94997999 0.92248789 0.94499862 0.93      ]
Media Macro-F1: 0.9379915039832019

Esecuzione: caratteri_3_5_grammi
Scores fold: [0.9799955  0.98749805 0.98249989 0.9874993  0.9874993 ]
Media Macro-F1: 0.9849984059704517


## 4. Addestramento Finale e Valutazione delle Performance

La configurazione basata su **n-grammi di caratteri (3-5)** ha ottenuto il punteggio di Cross-Validation più alto (~0.985). 

Procediamo ora ad addestrare il modello definitivo sull'intero Training Set per poi valutarlo in modo indipendente sul **Validation Set** e sul **Test Set**.


In [19]:
best_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(3, 5))),
    ("clf", LinearSVC())
])

# training finale
best_pipeline.fit(df_train['text'], df_train['label'])

# Predizioni su validation e test
pred_val = best_pipeline.predict(df_val['text'])
pred_test = best_pipeline.predict(df_test['text'])

print("=== Validation ===")
print(classification_report(df_val['label'], pred_val, zero_division=0))

print("=== Test ===")
print(classification_report(df_test['label'], pred_test, zero_division=0))

=== Validation ===
              precision    recall  f1-score   support

           0       0.97      0.98      0.98       500
           1       0.98      0.97      0.98       500

    accuracy                           0.98      1000
   macro avg       0.98      0.98      0.98      1000
weighted avg       0.98      0.98      0.98      1000

=== Test ===
              precision    recall  f1-score   support

           0       0.76      0.98      0.85       500
           1       0.98      0.68      0.80       500

    accuracy                           0.83      1000
   macro avg       0.87      0.83      0.83      1000
weighted avg       0.87      0.83      0.83      1000



### 📝 Considerazioni sui Risultati

* **Validation Set**: Il modello mostra performance eccellenti (Macro F1 = 0.98), in linea con quanto visto in Cross-Validation.
* **Test Set**: Si nota un calo significativo dell'accuratezza generale (0.83) e in particolare del recall sulla classe 1 (0.68). 

**Diagnosi**: Questo comportamento suggerisce la presenza di un potenziale **overfitting** o, più probabilmente, di un **domain shift** (discrepanza nella distribuzione dei dati) tra il set di train/validation e il test set. Le feature basate su n-grammi di caratteri potrebbero aver catturato pattern stilistici troppo specifici dei dati di addestramento.